In [ ]:
def solutoin(triangle):
    dp = [row[:] for row in triangle]
    for i in range(1, len(dp)):
        for j in range(i+1):
            left = dp[i-1][j-1] if j > 0 else 0
            right = dp[]i-1[j] if j < i else 0
            dp[i][j] += max(left, right)
    return max(dp[-1])

In [ ]:
def solutoin(t):
    row = len(t)
    col = len(t[-1])

    cost = [[0] * col for _ in range(row)]

    for i in range(row):
        for j in range(len(t[i])):
            if i == 0 and j == 0:
                cost[i][j] = t[i][j]
                continue
            if j != 0:
                cost[i][j] = max(cost[i-1][j], cost[i-1][j-1])+t[i][j]
            else:
                cost[i][j] = cost[i-1][j] + t[i][j]
    print(cost)
    answer = max(cost[-1])
    return answer

In [2]:
# [2권 Chapter 1. 머신러닝이란: 규칙을 짜지 않고 데이터로 배우기 · 1.3 머신러닝의 세 갈래 · manuscript-vol2/chapters/01-what-is-ml.md:240]
import numpy as np
from sklearn.cluster import KMeans
from sklearn.datasets import load_iris

X, y = load_iris(return_X_y=True)

# 레이블 y는 주지 않고 특성 X만 주면서 "비슷한 것끼리 세 묶음으로" 나누라고 한다
묶음 = KMeans(n_clusters=3, n_init=10, random_state=0).fit_predict(X)

# 다 나눈 뒤에야 실제 품종과 맞춰 본다
for 번호 in range(3):
    품종별 = np.bincount(y[묶음 == 번호], minlength=3)
    print(f"묶음 {번호}: setosa {품종별[0]:2d}, versicolor {품종별[1]:2d}, virginica {품종별[2]:2d}")


묶음 0: setosa  0, versicolor 48, virginica 14
묶음 1: setosa 50, versicolor  0, virginica  0
묶음 2: setosa  0, versicolor  2, virginica 36


In [ ]:
# [2권 Chapter 1. 머신러닝이란: 규칙을 짜지 않고 데이터로 배우기 · 1.5 첫 모델: 최근접 이웃 분류 · manuscript-vol2/chapters/01-what-is-ml.md:380]
import numpy as np
from sklearn.datasets import load_iris

X, y = load_iris(return_X_y=True)
새_꽃 = np.array([6.0, 2.9, 4.6, 1.5])   # 정답을 모르는 꽃 한 송이 (예제용)
print(type(X))
print(type(y))
print(X.shape)
print(새_꽃.shape)

차이 = X - 새_꽃                         # (150, 4) - (4,) → 브로드캐스팅  'numpy.ndarray' 간의 뺴기면 에러가 나는게아니고 에러를 가로채서 브로드캐스팅 연산을 해준다.
거리 = np.sqrt((차이 ** 2).sum(axis=1))  # 꽃마다 거리 하나 → (150,) -> 150, 4 로 나온걸 가로 150 세로 1로 만듬
print("거리 모양:", 거리.shape)

가까운_순 = np.argsort(거리)[:5]          # 가까운 다섯 송이의 번호
for 번호 in 가까운_순:
    print(f"{번호:3d}번 꽃  거리 {거리[번호]:.3f}  레이블 {y[번호]}")
print("다섯 이웃의 표:", np.bincount(y[가까운_순], minlength=3))


<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
(150, 4)
(4,)
거리 모양: (150,)
 78번 꽃  거리 0.100  레이블 1
 91번 꽃  거리 0.173  레이블 1
 63번 꽃  거리 0.173  레이블 1
 73번 꽃  거리 0.346  레이블 1
138번 꽃  거리 0.374  레이블 2
다섯 이웃의 표: [0 4 1]


In [ ]:
# [2권 Chapter 1. 머신러닝이란: 규칙을 짜지 않고 데이터로 배우기 · 1.5 첫 모델: 최근접 이웃 분류 · manuscript-vol2/chapters/01-what-is-ml.md:445]
import numpy as np
from sklearn.datasets import load_iris


def 최근접_이웃_예측(X_학습: np.ndarray, y_학습: np.ndarray,
                    X_새: np.ndarray, k: int) -> np.ndarray:
    """새 샘플마다 가장 가까운 학습 샘플 k개의 다수결로 레이블을 정한다."""
    # (새 개수, 1, 특성) - (1, 학습 개수, 특성) → (새 개수, 학습 개수, 특성)
    print(X_새.shape) #(30, 4)
    print(X_학습.shape) # (120, 4)

    차이 = X_새[:, np.newaxis, :] - X_학습[np.newaxis, :, :]

    print(X_새[:, np.newaxis, :].shape) # (30, 1, 4)
    print(X_학습[np.newaxis, :, :].shape) # (1, 120, 4)
    print(차이.shape) # (30, 120, 4) 이렇게 되는것도 브로드 캐스팅과 원리는 같다.



    거리 = np.sqrt((차이 ** 2).sum(axis=2))           # (새 개수, 학습 개수) 피타고라스 정리
    이웃 = np.argsort(거리, axis=1, kind="stable")[:, :k]  # 가까운 k개의 번호
    이웃_레이블 = y_학습[이웃]                         # (새 개수, k)
    return np.array([np.bincount(줄).argmax() for 줄 in 이웃_레이블])
# bincount 이웃에서 도출된거 숫자들을 카운팅 2 2 1 2 0 이면 0은 1회 1은 1회 2는 3회 이렇게 리스트화 [1, 1, 3] 그럼  argmax는 거기서 숫자게 제일 많은 3의 인덱스인 2을 추출


X, y = load_iris(return_X_y=True)
섞은_번호 = np.random.default_rng(42).permutation(len(X))
X_학습, y_학습 = X[섞은_번호[30:]], y[섞은_번호[30:]]
X_테스트, y_테스트 = X[섞은_번호[:30]], y[섞은_번호[:30]]

예측 = 최근접_이웃_예측(X_학습, y_학습, X_테스트, k=5)
print("예측:", 예측[:10])
print("정답:", y_테스트[:10])
print(f"테스트 정확도: {(예측 == y_테스트).mean():.3f} ({(예측 == y_테스트).sum()}/{len(y_테스트)})")
